# SynBioCrow 2.2.0-rc1 — All Generators Setup
This notebook installs/imports SynBioCrow plus DORAnet, RetroBioCat2, RetroPath2/rp2paths, RetroTide, and (optionally) BioPKS Pipeline.\n\n**Important:** importable does not mean execution-ready. RBC2 needs its data assets; RetroPath2 needs KNIME + RetroRules + sink data; BioPKS remains an external authorized runtime and requires explicit license acknowledgement.\n

In [ ]:
import os, sys, subprocess, json, time, shutil, importlib, textwrap
from pathlib import Path
T0=time.time()
def status(p,msg): print(f'[SynBioCrow RC1] {p:5.1f}% | elapsed {(time.time()-T0)/60:5.1f} min | {msg}',flush=True)
status(1,'environment')
print(sys.version)


## 1. Install SynBioCrow core + DORAnet + thermodynamics\nThis installs the consolidated branch and the two Python extras that are safe to install directly in the active notebook environment.\n

In [ ]:
repo=Path('/content/SynBioCrow')
if repo.exists(): shutil.rmtree(repo)
status(5,'clone SynBioCrow 2.2 consolidation branch')
subprocess.run(['git','clone','--depth','1','--branch','develop/2.2-engine-consolidation','https://github.com/theiman112860/SynBioCrow.git',str(repo)],check=True)
status(10,'install core + DORAnet + thermo')
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(repo)+'[doranet,thermo]'],check=True)


## 2. Install RetroBioCat2\nSynBioCrow uses RetroBioCat2 as a separate research runtime. This cell installs the proven upstream revision without letting it replace the whole scientific environment.\n

In [ ]:
status(20,'install RetroBioCat2 runtime dependencies')
subprocess.run([sys.executable,'-m','pip','install','-q','PyYAML','sqlalchemy','tables','gdown','tqdm'],check=True)
RBC2_REV='c5f32561b1ed99d0701a6dbb472dc448d6c082e8'
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','--no-build-isolation',f'git+https://github.com/willfinnigan/RetroBioCat-2.git@{RBC2_REV}'],check=True)
print('RetroBioCat2 installed. Real searches still require RBC2 scientific data assets.')


## 3. Install RetroPath2 wrapper + rp2paths\nThe Python packages can be installed here. Real RetroPath2 execution additionally requires KNIME, a RetroRules-ready rules file, and a sink file.\n

In [ ]:
status(35,'install RetroPath2 Python components')
subprocess.run([sys.executable,'-m','pip','install','-q','git+https://github.com/brsynth/retropath2-wrapper.git'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','git+https://github.com/brsynth/rp2paths.git'],check=True)
print('RetroPath2 Python components installed.')
print('Execution-ready still requires KNIME + RetroRules rules.csv + sink.csv.')


## 4. Install RetroTide\nRetroTide is installed separately as a specialized PKS generator.\n

In [ ]:
status(50,'install RetroTide')
subprocess.run([sys.executable,'-m','pip','install','-q','git+https://github.com/JBEI/RetroTide.git'],check=True)


## 5. BioPKS Pipeline in an isolated Python 3.10 environment
BioPKS Pipeline pins an older scientific stack (including DORAnet 0.5.6a1 and eQuilibrator <0.6), so it must **not** be installed into the main SynBioCrow environment. The upstream README recommends Python 3.10. This cell builds an isolated BioPKS runtime and keeps SynBioCrow connected through the external bridge contract.


In [ ]:
ACK_BIOPKS_LICENSE=False  # set True only after reviewing https://github.com/JBEI/BioPKS-Pipeline/blob/main/license.txt
BIOPKS_ROOT=Path('/content/BioPKS-Pipeline')
BIOPKS_VENV=Path('/content/biopks_py310')
BIOPKS_PYTHON=BIOPKS_VENV/'bin/python'

def run_live(cmd, *, timeout_s=900, label='subprocess'):
    import threading, queue
    q=queue.Queue()
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    def pump():
        for line in proc.stdout:
            q.put(line)
        q.put(None)
    threading.Thread(target=pump,daemon=True).start()
    start=time.time(); last=time.time(); done=False
    while True:
        try:
            item=q.get(timeout=1)
            if item is None:
                done=True
            else:
                print(item.rstrip(),flush=True); last=time.time()
        except queue.Empty:
            pass
        if done and proc.poll() is not None:
            break
        now=time.time()
        if now-last >= 30:
            print(f'[{label}] still running | elapsed {(now-start)/60:.1f} min',flush=True)
            last=now
        if now-start > timeout_s:
            proc.terminate()
            try: proc.wait(timeout=10)
            except subprocess.TimeoutExpired: proc.kill()
            raise TimeoutError(f'{label} exceeded {timeout_s/60:.1f} min')
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode,cmd)
    return proc.returncode

BIOPKS_STATUS={'configured':False,'execution_ready':False,'reason':'license not acknowledged'}
if ACK_BIOPKS_LICENSE:
    status(62,'prepare isolated BioPKS Python 3.10 runtime')
    if BIOPKS_ROOT.exists(): shutil.rmtree(BIOPKS_ROOT)
    run_live(['git','clone','--depth','1','https://github.com/JBEI/BioPKS-Pipeline.git',str(BIOPKS_ROOT)],timeout_s=300,label='BioPKS clone')
    # uv supplies an isolated Python 3.10 without disturbing the active Colab interpreter.
    subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
    run_live(['uv','python','install','3.10'],timeout_s=600,label='uv Python 3.10 install')
    if BIOPKS_VENV.exists(): shutil.rmtree(BIOPKS_VENV)
    run_live(['uv','venv','--python','3.10',str(BIOPKS_VENV)],timeout_s=300,label='BioPKS venv creation')
    run_live(['uv','pip','install','--python',str(BIOPKS_PYTHON),'-e',str(BIOPKS_ROOT)],timeout_s=1200,label='BioPKS isolated install')
    os.environ['SYNBIOCROW_BIOPKS_ACK_LICENSE']='1'
    os.environ['SYNBIOCROW_BIOPKS_PYTHON']=str(BIOPKS_PYTHON)
    BIOPKS_STATUS={'configured':True,'execution_ready':BIOPKS_PYTHON.exists(),'python':str(BIOPKS_PYTHON)}
else:
    print('BioPKS installation skipped. Review upstream license and set ACK_BIOPKS_LICENSE=True to install.')
print('[BIOPKS STATUS]',json.dumps(BIOPKS_STATUS,indent=2))


## 6. Import matrix\nThis checks Python importability independently from backend execution readiness.

In [ ]:
status(68,'restore SynBioCrow editable install after backend installs')
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(repo)],check=True)
if str(repo) not in sys.path:
    sys.path.insert(0,str(repo))
importlib.invalidate_caches()
spec=importlib.util.find_spec('synbiocrow')
print('[SYNBIOCROW SPEC]',spec)
subprocess.run([sys.executable,'-m','pip','show','synbiocrow'],check=False)
if spec is None:
    raise RuntimeError('SynBioCrow is still not importable after explicit reinstall + sys.path repair')

status(72,'import matrix')
MODULES={
 'synbiocrow':'synbiocrow',
 'DORAnet':'doranet',
 'RetroBioCat2':'rbc2',
 'RetroPath2 wrapper':'retropath2_wrapper',
 'rp2paths':'rp2paths',
 'RetroTide':'retrotide',
 'eQuilibrator':'equilibrator_api',
}
import_status={}
for label,name in MODULES.items():
    try:
        mod=importlib.import_module(name)
        import_status[label]={'importable':True,'module':getattr(mod,'__file__',None)}
    except Exception as exc:
        import_status[label]={'importable':False,'error':f'{type(exc).__name__}: {exc}'}

if ACK_BIOPKS_LICENSE and BIOPKS_PYTHON.exists():
    probe="import biopks_pipeline, json; print(json.dumps({'ok': True, 'module': getattr(biopks_pipeline, '__file__', None)}))"
    p=subprocess.run([str(BIOPKS_PYTHON),'-c',probe],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    import_status['BioPKS Pipeline']={'importable':p.returncode==0,'isolated_python':str(BIOPKS_PYTHON),'output':p.stdout[-2000:]}
else:
    import_status['BioPKS Pipeline']={'importable':False,'reason':'isolated runtime not installed'}
print(json.dumps(import_status,indent=2,sort_keys=True))


## 7. SynBioCrow backend readiness\nThis is stricter than importability and reports configuration/resource gaps.

In [ ]:
status(82,'SynBioCrow readiness')
from synbiocrow import SynBioCrowEngine
engine=SynBioCrowEngine()
print(json.dumps(engine.backend_readiness(),indent=2,sort_keys=True))
from synbiocrow.bootstrap import bootstrap_advice
for row in bootstrap_advice(engine):
    print(row)


## 8. Configure external resources\nSet these only when you have the actual resources. No placeholder path is treated as execution-ready.\n

In [ ]:
RETROPATH_RULES_FILE=None   # e.g. '/content/drive/MyDrive/.../rules.csv'
RETROPATH_SINK_FILE=None    # e.g. '/content/drive/MyDrive/.../sink.csv'
BIOPKS_BRIDGE=None          # bridge script inside/alongside authorized BioPKS runtime
if ACK_BIOPKS_LICENSE and BIOPKS_PYTHON.exists():
    os.environ['SYNBIOCROW_BIOPKS_PYTHON']=str(BIOPKS_PYTHON)
if BIOPKS_BRIDGE:
    os.environ['SYNBIOCROW_BIOPKS_RUNNER']=BIOPKS_BRIDGE
print('RetroPath execution-ready:', bool(RETROPATH_RULES_FILE and RETROPATH_SINK_FILE))
print('BioPKS isolated Python:', os.environ.get('SYNBIOCROW_BIOPKS_PYTHON'))
print('BioPKS bridge configured:', bool(BIOPKS_BRIDGE))


## 9. Run M7/M9 core verification\nThis does not access protected benchmark truth.

In [ ]:
status(90,'run release-readiness gate')
subprocess.run([sys.executable,str(repo/'scripts/m7_release_readiness.py')],check=True)
status(96,'run M9 package check')
subprocess.run([sys.executable,str(repo/'scripts/build_m9_release.py'),'--check'],check=True)
status(100,'all-generators environment setup complete')
